# DateTime and Text in Pandas

A runnable walkthrough of parsing dates, extracting features, filtering time windows, time zones, and cleaning text.

In [1]:
import pandas as pd
pd.__version__

'2.2.3'

## 1. Parse dates, including invalid values

In [2]:
raw_dates = pd.Series(["2026-01-05", "2026-02-17", "not-a-date"])
parsed_dates = pd.to_datetime(raw_dates, errors="coerce")
pd.DataFrame({"raw": raw_dates, "parsed": parsed_dates})

,raw,parsed
0,2026-01-05,2026-01-05
1,2026-02-17,2026-02-17
2,not-a-date,NaT


## 2. Parse a known date format

In [3]:
day_first = pd.Series(["14/10/2026", "25/12/2026"])
pd.to_datetime(day_first, format="%d/%m/%Y")

0   2026-10-14
1   2026-12-25
dtype: datetime64[ns]

## 3. Extract date components

In [4]:
orders = pd.DataFrame({"date": ["2026-01-31", "2026-02-01", "2026-03-15"]})
orders["date"] = pd.to_datetime(orders["date"])
orders["year"] = orders["date"].dt.year
orders["month"] = orders["date"].dt.month
orders["weekday"] = orders["date"].dt.day_name()
orders["is_month_end"] = orders["date"].dt.is_month_end
orders

,date,year,month,weekday,is_month_end
0,2026-01-31,2026,1,Saturday,True
1,2026-02-01,2026,2,Sunday,False
2,2026-03-15,2026,3,Sunday,False


## 4. Filter a whole month safely

In [5]:
events = pd.DataFrame({"timestamp": pd.to_datetime([
    "2026-01-01 00:00", "2026-01-31 23:59", "2026-02-01 00:00"
])})
january = events.loc[
    (events["timestamp"] >= "2026-01-01") &
    (events["timestamp"] < "2026-02-01")
]
january

,timestamp
0,2026-01-01 00:00:00
1,2026-01-31 23:59:00


## 5. Timedeltas and date ranges

In [6]:
deliveries = pd.DataFrame({
    "ordered": pd.to_datetime(["2026-01-01", "2026-01-03"]),
    "delivered": pd.to_datetime(["2026-01-04", "2026-01-08"])
})
deliveries["days_to_deliver"] = (deliveries["delivered"] - deliveries["ordered"]).dt.days
calendar = pd.date_range("2026-01-01", periods=5, freq="D")
deliveries, calendar

(     ordered  delivered  days_to_deliver
 0 2026-01-01 2026-01-04                3
 1 2026-01-03 2026-01-08                5,
 DatetimeIndex(['2026-01-01', '2026-01-02', '2026-01-03', '2026-01-04',
                '2026-01-05'],
               dtype='datetime64[ns]', freq='D'))

## 6. Time-zone conversion

In [7]:
utc_times = pd.Series(pd.to_datetime(["2026-01-01 09:00:00", "2026-06-01 12:30:00"], utc=True))
india_times = utc_times.dt.tz_convert("Asia/Kolkata")
pd.DataFrame({"utc": utc_times, "india": india_times})

,utc,india
0,2026-01-01 09:00:00+00:00,2026-01-01 14:30:00+05:30
1,2026-06-01 12:30:00+00:00,2026-06-01 18:00:00+05:30


## 7. Clean text

In [8]:
customers = pd.DataFrame({"name": ["  ALICE smith ", "bob JONES", None], "product": ["Pro Laptop", "basic mouse", "PRO monitor"]})
customers["name_clean"] = customers["name"].astype("string").str.strip().str.title()
customers["product_lower"] = customers["product"].astype("string").str.lower()
customers["contains_pro"] = customers["product"].astype("string").str.contains("pro", case=False, na=False)
customers

,name,product,name_clean,product_lower,contains_pro
0,ALICE smith,Pro Laptop,Alice Smith,pro laptop,True
1,bob JONES,basic mouse,Bob Jones,basic mouse,False
2,None,PRO monitor,<NA>,pro monitor,True


## 8. Split and extract structured text

In [9]:
tickets = pd.DataFrame({
    "ticket": ["BUG-102", "REQ-204", "BUG-305"],
    "email": ["ana@example.com", "ben@sample.org", "cy@example.com"]
})
tickets["ticket_type"] = tickets["ticket"].str.extract(r"^([A-Z]+)-")
tickets[["username", "domain"]] = tickets["email"].str.extract(r"(?P<username>[^@]+)@(?P<domain>[^@]+)")
tickets

,ticket,email,ticket_type,username,domain
0,BUG-102,ana@example.com,BUG,ana,example.com
1,REQ-204,ben@sample.org,REQ,ben,sample.org
2,BUG-305,cy@example.com,BUG,cy,example.com


## 9. Literal and regex replacement

In [10]:
codes = pd.Series(["INV-12345", "ORD-007", "INV-908"])
literal_replaced = codes.str.replace("-", " ", regex=False)
masked_numbers = codes.str.replace(r"\d+", "#", regex=True)
pd.DataFrame({"original": codes, "literal_replaced": literal_replaced, "masked_numbers": masked_numbers})

,original,literal_replaced,masked_numbers
0,INV-12345,INV 12345,INV-#
1,ORD-007,ORD 007,ORD-#
2,INV-908,INV 908,INV-#


## 10. Integrated cleaning workflow

In [11]:
df = pd.DataFrame({
    "created": ["2026-01-02", "bad-date", "2026-01-20"],
    "customer": ["  ALICE ", "Bob", "carol  "],
    "ticket": ["BUG-102", "REQ-204", "BUG-305"]
})
df["created"] = pd.to_datetime(df["created"], errors="coerce")
invalid_date_count = int(df["created"].isna().sum())
df["customer"] = df["customer"].astype("string").str.strip().str.title()
df["ticket_type"] = df["ticket"].astype("string").str.extract(r"^([A-Z]+)-")
df["month"] = df["created"].dt.month
df, invalid_date_count

(     created customer   ticket ticket_type  month
 0 2026-01-02    Alice  BUG-102         BUG    1.0
 1        NaT      Bob  REQ-204         REQ    NaN
 2 2026-01-20    Carol  BUG-305         BUG    1.0,
 1)